# Act II — Fine-tune Qwen2.5-0.5B into a Shakespearean rewriter (free GPU)

Runs on a **free** Colab or Kaggle T4. We LoRA/QLoRA fine-tune a small open-source instruct
model on the style-transfer dataset built by `build_dataset.py`, so it learns to rewrite modern
English in Shakespeare's voice.

**Why LoRA?** The base model already speaks English. We only need to teach it a *style shift*,
so we freeze the 0.5B base weights and train a few million low-rank adapter parameters. That fits
in free-tier GPU memory and trains in minutes.

**Runtime:** set Runtime → Change runtime type → **T4 GPU** before running.

In [ ]:
# 1. Install the fine-tuning stack. Pinned-ish to versions known to play together.
!pip -q install "transformers>=4.44" "peft>=0.12" "trl>=0.9" "datasets>=2.20" \
    "bitsandbytes>=0.43" "accelerate>=0.33"

In [ ]:
# 2. Get the dataset. Either clone this repo, or upload data/train.jsonl + data/val.jsonl.
#    (Clone is simplest once this repo is on GitHub.)
# !git clone https://github.com/<you>/bardbox && cp bardbox/module2_finetune/data/*.jsonl .
from google.colab import files  # noqa
# Upload train.jsonl and val.jsonl if not cloning:
# uploaded = files.upload()

from datasets import load_dataset
ds = load_dataset("json", data_files={"train": "train.jsonl", "validation": "val.jsonl"})
print(ds)
print(ds["train"][0])

In [ ]:
# 3. Load the base model in 4-bit (QLoRA) + its tokenizer.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE = "Qwen/Qwen2.5-0.5B-Instruct"  # ungated, tiny, has a chat template

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
tok = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map="auto")

In [ ]:
# 4. BEFORE fine-tuning: see how the base model rewrites. We'll compare after.
def rewrite(model, text, max_new_tokens=60):
    msgs = [
        {"role": "system", "content": "You are a Shakespearean playwright. Rewrite the user's modern English in Shakespeare's style."},
        {"role": "user", "content": text},
    ]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = tok(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.7)
    return tok.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True)

print("BEFORE:", rewrite(model, "I am very tired and I want to go home."))

In [ ]:
# 5. Attach a LoRA adapter and fine-tune with trl's SFTTrainer.
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

cfg = SFTConfig(
    output_dir="bardbox-qwen-lora",
    num_train_epochs=8,          # small dataset -> several passes
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=5,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    fp16=True,
)

trainer = SFTTrainer(
    model=model, args=cfg, train_dataset=ds["train"],
    eval_dataset=ds["validation"], peft_config=lora,
)
trainer.train()

In [ ]:
# 6. AFTER fine-tuning: same prompts, compare to the BEFORE output above.
trainer.model.eval()
for t in ["I am very tired and I want to go home.",
          "Please stop lying to me.",
          "The weather is beautiful today."]:
    print(t, "->", rewrite(trainer.model, t))

trainer.save_model("bardbox-qwen-lora")  # saves the adapter

In [ ]:
# 7. Merge the adapter into the base and save a standalone model for GGUF conversion.
#    (Then follow merge_and_quantize.md locally to make the Q4 GGUF for Act III.)
from peft import AutoPeftModelForCausalLM

merged = AutoPeftModelForCausalLM.from_pretrained("bardbox-qwen-lora").merge_and_unload()
merged.save_pretrained("bardbox-qwen-merged")
tok.save_pretrained("bardbox-qwen-merged")
print("Saved merged model. Download bardbox-qwen-merged/ for local GGUF conversion.")